# Chapter 9 §9.3: Marketing Content Generator

Style references ground generation and supply an embedding-based signal for style-consistency optimization.


In [ ]:
%pip install -r ../requirements.txt -q


## Runtime setup

The cells tagged `chapter-example` reproduce the manuscript code blocks verbatim. Supporting cells only provide imports, fixtures, or safe local stand-ins for external services.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import dspy

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in ../.env before running this notebook.")

lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)

import numpy as np

embedder = dspy.Embedder(
    "openai/text-embedding-3-small",
    api_key=api_key,
)
dspy.configure(lm=lm, embedder=embedder)


## Style definitions and generation signature


In [ ]:
STYLES = {
    "technical": "Technical, precise, data-driven. Use specific "
                 "metrics. Short sentences. Avoid hype.",
    "conversational": "Friendly, approachable, casual. Use questions. "
                      "Include contractions. Conversational tone.",
    "executive": "Strategic, high-level, business-focused. "
                 "Emphasize ROI. Professional but accessible."
}


In [ ]:
class StyleAwareContentGenerator(dspy.Signature):
    """Write marketing blog content matching a specific style."""

    topic: str = dspy.InputField(desc="Blog topic or headline")
    style_guide: str = dspy.InputField(
        desc="Style characteristics to match"
    )
    content: str = dspy.OutputField(
        desc="Blog paragraph (2-3 sentences)"
    )


## Reference passages


In [ ]:
STYLE_REFERENCES = {
    "technical": [
        "Performance benchmarks show 99.9% uptime. "
        "Latency: p50=12ms, p99=45ms.",
        "The system processes 50,000 events/second "
        "with zero data loss.",
    ],
    "conversational": [
        "Want to ship faster? Our platform makes "
        "deployment a breeze.",
        "Here's the thing: your users expect speed. "
        "We deliver it.",
    ],
}


The printed dictionary is intentionally short. Add executive references so every key in `STYLES` has at least one evaluation target.


In [ ]:
STYLE_REFERENCES["executive"] = [
    "This investment reduces operating cost while improving time to market.",
    "The rollout ties platform reliability to measurable revenue outcomes.",
]


## Style transfer signature


In [ ]:
class StyleTransfer(dspy.Signature):
    """Transform content from one style to another
    while preserving meaning."""

    original_content: str = dspy.InputField(
        desc="Content to transform"
    )
    target_style: str = dspy.InputField(
        desc="Target style characteristics"
    )
    transformed_content: str = dspy.OutputField(
        desc="Content rewritten in target style"
    )


## Semantic-distance metric


In [ ]:
def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denominator == 0 else float(np.dot(a, b) / denominator)


In [ ]:
def style_similarity(content, reference_texts, embedder):
    content_embedding = embedder([content])[0]
    reference_embeddings = embedder(reference_texts)

    similarities = [
        cosine_similarity(content_embedding, ref_emb)
        for ref_emb in reference_embeddings
    ]
    return sum(similarities) / len(similarities)


In [ ]:
def style_consistency_metric(example, pred, trace=None):
    style_type = None
    for name, guide in STYLES.items():
        if guide == example.style_guide:
            style_type = name
            break

    refs = STYLE_REFERENCES.get(style_type, [])
    similarity = style_similarity(pred.content, refs, embedder)
    return max(0.0, min(1.0, similarity))
